In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.utils import to_categorical
import tarfile
import pickle
import numpy as np

def load_cifar10(tar_path='cifar-10-batches-py.tar.gz'):
    with tarfile.open(tar_path, 'r:gz') as tar:
        def get_batch(name):
            member = [m for m in tar.getmembers() if m.name.endswith(name)][0]
            return pickle.load(tar.extractfile(member), encoding='bytes')

        # Load 5 training batches
        x_tr, y_tr = [], []
        for i in range(1, 6):
            b = get_batch(f'data_batch_{i}')
            x_tr.append(b[b'data'])
            y_tr.extend(b[b'labels'])

        # Load test batch
        b_test = get_batch('test_batch')
        x_te, y_te = b_test[b'data'], b_test[b'labels']

    # Reshape from flat vectors (3072,) -> (32, 32, 3) image format
    X_train = np.vstack(x_tr).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_train = np.array(y_tr).reshape(-1, 1)

    X_test = x_te.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_test = np.array(y_te).reshape(-1, 1)

    return (X_train, y_train), (X_test, y_test)

(X_train, y_train), (X_test, y_test) = load_cifar10('cifar-10-python.tar.gz')

In [ ]:
#step 2 - Resize for MobileNetV2 (224x224)
from tensorflow.image import resize

X_train = tf.image.resize(X_train, [224, 224])
X_test = tf.image.resize(X_test, [224, 224])

In [ ]:
#step 3 - Load MobileNetV2 without top classification head
from tensorflow.keras.applications import MobileNetV2

#load MobileNetV2 base model
base_model = MobileNetV2(weights='imagenet',
                         include_top=False, #without top classification head
                         input_shape=(224, 224, 3))

base_model.trainable = False  #going with freezed pretrained layers (for now)

In [ ]:
from akida_models import fetch_file

model = base_model

samples = X_train[:1024].numpy().astype(np.float32)

In [ ]:
from quantizeml.models import QuantizationParams
qparams4 = QuantizationParams(input_weight_bits=8, weight_bits=4, activation_bits=4)

from quantizeml.models import quantize
quantized_model = quantize(model, qparams=qparams4, samples=samples)

from cnn2snn import convert
akida_model = convert(quantized_model)

In [ ]:
import numpy as np
import akida

# Conversion check only (accuracy is meaningless until the model has a trained head).
akida_model.summary()
print("input shape :", akida_model.input_shape)
print("output shape:", akida_model.output_shape)

# Run a few images through the simulator: if this works and shapes match, conversion is fine.
x = X_test[:8].numpy().astype(np.uint8)
out = np.asarray(akida_model.forward(x))
ref = quantized_model.predict(x.astype(np.float32), verbose=0)
print("akida out:", out.shape, "| quantized keras out:", ref.shape)

print("devices:", akida.devices())